# **Lab 6 Transfer Learning & Hyperparameter Tuning**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader,Subset,Dataset
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import os
import cv2
from skimage.util import random_noise
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

seed = 4912
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data Preparation
Complete the class `CustomImageDataset()` that `__getitem__` return ***noisy blury*** image and ***ground truth*** image.
Please ensure that the final image is in RGBscale and has a size of 128x128.

In [2]:
### START CODE HERE ###
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, gauss_noise=False, gauss_blur=None, resize=128, p=0.5):
        self.p = p
        self.resize = resize
        self.gauss_noise = gauss_noise
        self.gauss_blur = gauss_blur
        self.image_paths = image_paths

        self.base_transform = transforms.Compose([
            transforms.Resize((resize, resize)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def add_gaussian_noise(self, image):
        mean = random.uniform(-50, 50)
        std = random.uniform(5, 25)
        np_img = np.array(image).astype(np.float32)

        noise = np.random.normal(mean, std, np_img.shape)
        noisy_img = np_img + noise

        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy_img)

    def add_gaussian_blur(self, image):
        kernel_size = random.randrange(3, 12, 2)
        np_img = np.array(image)
        blurred = cv2.GaussianBlur(np_img, (kernel_size, kernel_size), 0)
        return Image.fromarray(blurred)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        gt_image = Image.open(path).convert("RGB")
        gt_image = self.base_transform(gt_image)

        image = Image.open(path).convert("RGB")

        if self.gauss_blur and random.random() < self.p:
            image = self.add_gaussian_blur(image)

        if self.gauss_noise and random.random() < self.p:
            image = self.add_gaussian_noise(image)

        image = self.base_transform(image)

        return image, gt_image
### END CODE HERE ###

In [3]:
### START CODE HERE ###
def imshow_grid(images):
    num_images = len(images)
    num_rows = int(np.ceil(np.sqrt(num_images)))
    num_cols = int(np.ceil(num_images / num_rows))

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 6),dpi=100)
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            image = images[i].permute(1, 2, 0)
            ax.imshow(torch.clamp(image,0,1),cmap='gray')
            # ax.set_title(class_names[class_names.index(label)])
            ax.axis('off')
        else:
            ax.axis('off')  # Turn off empty subplots
    plt.tight_layout()
    plt.show()
### END CODE HERE ###

## Data Preparation
Complete the class `CustomImageDataset()` that `__getitem__` return ***noisy blury*** image and ***ground truth*** image.
Please ensure that the final image is in RGBscale and has a size of 128x128.

In [4]:
### START CODE HERE ###

torch.manual_seed(4912)
data_dir = 'data/img_align_celeba'

files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files]
# Split the dataset into training and testing sets
train_files, test_files = train_test_split(
    files, test_size=0.3, shuffle=True, random_state=2024)
print(files[0])
dataset = CustomImageDataset(train_files,
                            resize=128,
                            gauss_blur=True,
                            gauss_noise=True,
                            p=0.5
                            )
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=1)
### END CODE HERE ###

data/img_align_celeba\000001.jpg


In [ ]:
### START CODE HERE ###
batch,gt_img = next(iter(dataloader)) 

# print(batch)
# imshow_grid(batch)
# imshow_grid(gt_img)
### END CODE HERE ###

## Create Autoencoder model
You can design your own Autoencoder model based on the provided code below. However, please maintain the concept of 'Autoencoder'.

In [ ]:
### START CODE HERE ###
class DownSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DownSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class UpSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(UpSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.upsample(x)
        return x


class Autoencoder(nn.Module):
    def __init__(self, channels=[64, 128, 256], input_channels=3, output_channels=3):
        super().__init__()
        self.downsampling_blocks = nn.ModuleList()
        self.upsampling_blocks = nn.ModuleList()

        self.conv_in = nn.Conv2d(input_channels, channels[0], kernel_size=3, stride=1, padding=1)

        # Downsampling
        in_channels = channels[0]
        for out_channels in channels[1:]:
            self.downsampling_blocks.append(DownSamplingBlock(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            in_channels = out_channels

        # Upsampling
        for out_channels in reversed(channels[:-1]):
            self.upsampling_blocks.append(UpSamplingBlock(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            in_channels = out_channels

        self.conv_out = nn.Conv2d(in_channels, output_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv_in(x)

        down_outputs = []
        for down_block in self.downsampling_blocks:
            x = down_block(x)
            down_outputs.append(x)

        for up_block in self.upsampling_blocks:
            x = up_block(x)

        x = self.conv_out(x)
        return x

### END CODE HERE ###
        

## Train Autoencoder
Complete the `train()` function in the cell below. This function should evaluate the model at every epoch, log the ***training loss, test loss,test PSNR, test SSIM***. Additionally, it should save the model at the last epoch.
<details>
<summary>
<font size="3" color="orange">
<b>Expected output</b>
</font>
</summary>

- The log should resemble this, but not be identical

```
🤖Training on cuda
🚀Training Epoch [1/1]: 100%|██████████| 1313/1313 [01:45<00:00, 12.41batch/s, loss=0.0102] 
📄Testing: 100%|██████████| 563/563 [01:10<00:00,  7.95batch/s, loss=0.0106, psnr=16.7, ssim=0.348] 
Summary :
	Train	avg_loss: 0.017262999383663165
	Test	avg_loss: 0.010476540363861867 
                PSNR : 16.839487147468034 
                SSIM : 0.36090552368883694
...
```

</details>

Resource : [PyTorch Training loop](<https://pytorch.org/tutorials/beginner/introyt/trainingyt.html#:~:text=%3D0.9)-,The%20Training%20Loop,-Below%2C%20we%20have>), [PSNR & SSIM](https://ieeexplore.ieee.org/document/5596999)

In [ ]:
### START CODE HERE ###
def train(model,opt,loss_fn,train_loader,test_loader,epochs=10,writer=None,checkpoint_path=None,device='cpu'):
    print("🤖Training on", device)
    if checkpoint_path is not None:
        os.makedirs(checkpoint_path, exist_ok=True)
    model = model.to(device)
    # if writer is not None:
    #     writer.add_graph(model, next(iter(train_loader))[0].to(device))
    step = 0
    for epoch in range(epochs):
        
        model.train()
        train_bar = tqdm(train_loader,desc=f'🚀Training Epoch [{epoch+1}/{epochs}]',unit='batch')
        avg_train_loss = 0
        avg_test_loss = 0
        avg_train_acc = 0
        avg_test_acc = 0
        for images, gt in train_bar:
            images= images.to(device)
            gt = gt.to(device)
            opt.zero_grad()
            output = model(images)
            loss = loss_fn(output, gt)
            loss.backward()
            opt.step()
            avg_train_loss += loss.item()
            step +=1
            
            train_bar.set_postfix(loss=loss.item())
        avg_train_loss /= len(train_loader)

        total_psnr = 0
        total_ssim = 0
        model.eval()
        test_bar = tqdm(test_loader,desc='📄Testing',unit='batch')
        for images, gt in test_bar:
            images= images.to(device)
            gt = gt.to(device)
            with torch.no_grad():
                output = model(images)
                loss = loss_fn(output, gt)
                psnr_value = psnr(gt.detach().cpu().numpy(), images.detach().cpu().numpy(),
                                          data_range=1.0)
                ssim_value = ssim(gt.detach().cpu().numpy(), images.detach().cpu().numpy(),
                                    data_range=1.0, multichannel=True, win_size=3, win_size_y=3, win_size_z=1)
            psnr_values = []
            ssim_values = []
            for gt, img in zip(gt.detach().cpu().numpy(), images.detach().cpu().numpy()):
                psnr_value = psnr(gt, img, data_range=1.0)
                psnr_values.append(psnr_value)

                ssim_value = ssim(gt, img, data_range=1.0, multichannel=True, win_size=3, win_size_y=3, win_size_z=1)
                ssim_values.append(ssim_value)
            psnr_value = np.mean(psnr_values)
            ssim_value = np.mean(ssim_values)
            total_psnr += psnr_value
            total_ssim += ssim_value
            avg_test_loss += loss.item()

            test_bar.set_postfix(loss=loss.item(),psnr=psnr_value,ssim=ssim_value)
        avg_psnr = total_psnr / len(test_loader)
        avg_ssim = total_ssim / len(test_loader)
        avg_test_loss /= len(test_loader)
        if checkpoint_path is not None:
            state_dict = {
                        'optimizer': opt.state_dict(), 
                        'model': model.state_dict()
                          }
            torch.save(state_dict, f'{checkpoint_path}/model_ep{epoch}.pth')
        print(f"Summary :")
        print(f"\tTrain\tavg_loss: {avg_train_loss}")
        print(f"\tTest\tavg_loss: {avg_test_loss} \n\t\tPSNR : {avg_psnr} \n\t\tSSIM : {avg_ssim}")
### END CODE HERE ###

Let's train your model with 2 epochs to verify that your train() function works properly. After that, we'll move on to the Hyperparameter Grid Search in the next part.

In [ ]:
torch.manual_seed(4912)
data_dir = '/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba'

files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files]
# Split the dataset into training and testing sets
train_files, test_files = train_test_split(
    files, test_size=0.3, shuffle=True, random_state=2024)
print(files[0])
train_dataset = CustomImageDataset(train_files,
                            resize=132,
                            pad=((132,132),(132,132),(0,0)),
                            padding_mode='reflect',
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
test_dataset = CustomImageDataset(test_files,
                            resize=132,
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
testloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=16)

/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba/000001.jpg


In [ ]:
### START CODE HERE ###
model = Autoencoder()
print(model)
batch[0].unsqueeze(dim=0)
out = model(batch[0].unsqueeze(dim=0).float())
### END CODE HERE ###

Autoencoder(
  (downsampling_blocks): ModuleList(
    (0): DownSamplingBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): DownSamplingBlock(
      (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (upsampling_blocks): ModuleList(
    (0): UpSamplingBlock(
      (conv): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (upsample): Upsample(scale_factor=2.0, mode='b

In [ ]:
### START CODE HERE ###
checkpoint_path = 'AE'
# writer = SummaryWriter(f'{checkpoint_path}/logs')
writer = None
model = Autoencoder()
opt = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
train(model, opt, loss_fn, trainloader, testloader, epochs=1,checkpoint_path='AE',writer=writer, device='cuda')
### END CODE HERE ###

🤖Training on cuda


📄Testing: 100%|██████████| 563/563 [01:09<00:00,  8.07batch/s, loss=0.00903, psnr=18.9, ssim=0.51] 


Summary :
	Train	avg_loss: 0.017210818102238564
	Test	avg_loss: 0.010742807747600981 
		PSNR : 19.165548584006192 
		SSIM : 0.4797116545554291


---

## **Hyperparameter Grid Search with Raytune**

*If you have access to APEX, I would recommend converting this part into a Python file and submitting the job to run on APEX using SBATCH. This process may take a considerable amount of time.*

You can import additional Ray Tune tools as you want, such as schedulers, search algorithms, etc. Further information on the usage of Ray Tune can be found [here](https://docs.ray.io/en/latest/tune/index.html).

In [ ]:
import ray
from ray import tune
from ray.air import session


ray.shutdown()

/home/nouzen/miniconda3/envs/torch-ray_py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-11 20:58:40,479	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-09-11 20:58:40,982	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Complete the `train_raytune()` function below, following the [quick start guide](https://docs.ray.io/en/latest/tune/index.html). This function will be passed to the `tune.Tuner`.

In [ ]:
def train_raytune(config):
    torch.manual_seed(4912)

    data_dir = '/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba'
    files = os.listdir(data_dir)
    files = [os.path.join(data_dir, file) for file in files]
    train_files, test_files = train_test_split(
        files, test_size=0.3, shuffle=True, random_state=2024)
    train_dataset = CustomImageDataset(train_files,
                                resize=132,
                                pad=((132,132),(132,132),(0,0)),
                                padding_mode='reflect',
                                gauss_blur=True,
                                gauss_noise=True,
                                center_crop=128,
                                p=0.5
                                )
    test_dataset = CustomImageDataset(test_files,
                                resize=132,
                                gauss_blur=True,
                                gauss_noise=True,
                                center_crop=128,
                                p=0.5
                                )
    trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
    testloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=16)
    

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = Autoencoder()
    model = model.to(device)
    loss_fn = nn.MSELoss()


    if config['optimizer'] == 'Adam':
        opt = torch.optim.Adam(model.parameters(), lr=config['lr'])
    elif config['optimizer'] == 'SGD':
        opt = torch.optim.SGD(model.parameters(), lr=config['lr'])

    # config['optimizer'](model.parameters(),lr=config['lr'])

    for epoch in range(config['num_epochs']):
        model.train()
        # train_bar = tqdm(train_loader,desc=f'🚀Training Epoch [{epoch+1}/{epochs}]',unit='batch')
        avg_train_loss = 0
        avg_test_loss = 0
        for images, gt in trainloader:
            images= images.to(device)
            gt = gt.to(device)
            opt.zero_grad()
            output = model(images)
            loss = loss_fn(output, gt)
            loss.backward()
            opt.step()
            avg_train_loss += loss.item()
        avg_train_loss /= len(trainloader)

        total_psnr = 0
        total_ssim = 0
        model.eval()
        # test_bar = tqdm(test_loader,desc='📄Testing',unit='batch')
        for images, gt in testloader:
            images= images.to(device)
            gt = gt.to(device)
            with torch.no_grad():
                output = model(images)
                loss = loss_fn(output, gt)
                psnr_value = psnr(gt.detach().cpu().numpy(), images.detach().cpu().numpy(),
                                          data_range=1.0)
                ssim_value = ssim(gt.detach().cpu().numpy(), images.detach().cpu().numpy(),
                                    data_range=1.0, multichannel=True, win_size=3, win_size_y=3, win_size_z=1)
            psnr_values = []
            ssim_values = []
            for gt, img in zip(gt.detach().cpu().numpy(), images.detach().cpu().numpy()):
                psnr_value = psnr(gt, img, data_range=1.0)
                psnr_values.append(psnr_value)

                ssim_value = ssim(gt, img, data_range=1.0, multichannel=True, win_size=3, win_size_y=3, win_size_z=1)
                ssim_values.append(ssim_value)
            psnr_value = np.mean(psnr_values)
            ssim_value = np.mean(ssim_values)
            total_psnr += psnr_value
            total_ssim += ssim_value
            avg_test_loss += loss.item()
        avg_psnr = total_psnr / len(testloader)
        avg_ssim = total_ssim / len(testloader)
        avg_test_loss /= len(testloader)        


        session.report({
            "train_loss": avg_train_loss,
            "val_loss": avg_test_loss,
            "val_psnr": avg_psnr,
            "val_ssim": avg_ssim,
        })
        


Initialize Ray, define the search space, and resources.

Resource : 
- [A Guide To Parallelism and Resources for Ray Tune](https://docs.ray.io/en/latest/tune/tutorials/tune-resources.html#:~:text=A%20Guide%20To%20Parallelism%20and%20Resources%20for%20Ray%20Tune) 
- [Working with Tune Search Spaces](https://docs.ray.io/en/latest/tune/tutorials/tune-search-spaces.html#tune-search-space-tutorial:~:text=Working%20with%20Tune%20Search%20Spaces)
- [How to configure logging in Tune?](https://docs.ray.io/en/latest/tune/tutorials/tune-output.html) 
- [Tune Trial Schedulers (`tune.schedulers`)](https://docs.ray.io/en/latest/tune/api/schedulers.html#tune-scheduler-pbt:~:text=Tune%20Trial...-,Tune%20Trial%20Schedulers%20(tune.schedulers),-%23)

**Search Space:**
- `architecture`:<br>
    Feature map dimensions for convolutional layers<br>
    - `[32, 64, 128]`: 3 downsampling layers with feature maps increasing from 32 to 128.
    - `[64, 128, 256]`: 3 downsampling layers with feature maps starting from 64 to 256.
    - `[64, 128, 256, 512]`: 4 downsampling layers with more depth, starting from 64 and ending at 512.
- `learning rates (lr)`:
    - [1e-3, 8e-4, 1e-4, 1e-2]: Test a wide range of learning rates to evaluate model performance, from 1e-3 (typical) to a more aggressive 1e-2 or conservative 1e-4.
- `batch size`:
    - [16, 32]: Explore smaller batch sizes to evaluate their impact on gradient estimation and memory usage.
- `number of epochs`:
    - `[10, 50, 100]`: Allow short and long training sessions, from quick evaluations (10 epochs) to more extensive training (100 epochs).
- `optimizers (opts)`:
    - `["Adam", "SGD"]`: Compare two popular optimization algorithms: Adam for adaptive learning rates and SGD for momentum-based updates.

In [ ]:
ray.init(num_gpus=1,)

architecture = [[32,64,128],[64,128,256],[64,128,256,512]]
lr = [1e-3,8e-4,1e-4,1e-2]
batch_size = [16,32]
num_epochs = [10,50,100]
opts = ["Adam","SGD"]
# opts = [torch.optim.Adam,torch.optim.SGD]

# Customizable
config = {
    'architecture': tune.grid_search(architecture),
    'optimizer': tune.grid_search(opts),
    'lr': tune.grid_search(lr),
    'batch_size': tune.grid_search(batch_size),
    'num_epochs': tune.grid_search(num_epochs),
}

# search_alg = ConcurrencyLimiter(OptunaSearch(metric=["val_loss","val_accuracy"],mode=["min","max"]), max_concurrent=1)

# 1 GPU/12GB 
# if "gpu" = 1 -> 1 trial : 12GB -> 1 trial at a time
# 1 GPU/12GB 
# if "gpu" = 0.5 -> 1 trial : 6GB -> 2 trial at a time

# 2 GPU/12GB 
# if "gpu" = 1 -> 1 trial : 12GB -> 2 trial at a time
# 2 GPU/12GB 
# if "gpu" = 0.5 -> 1 trial : 6GB -> 4 trial at a time
tuner = tune.Tuner(  # ③
    tune.with_resources(train_raytune, resources={"gpu": 0.5}),
    tune_config=tune.TuneConfig(
        metric="val_psnr",
        mode="max",
        # scheduler=ASHAScheduler(grace_period=10,
        #                         reduction_factor=3,),
    ),
    param_space=config,
)
result = tuner.fit()

2024-09-11 13:37:44,605	INFO tune.py:1016 -- Wrote the latest version of all result files and experiment state to '/home/nouzen/ray_results/train_raytune_2024-09-11_13-28-54' in 0.0026s.
2024-09-11 13:37:44,609	INFO tune.py:1048 -- Total run time: 530.26 seconds (530.24 seconds for the tuning loop).


In [ ]:
path = "/home/nouzen/ray_results/train_raytune_2024-09-10_10-59-31"
restored_tuner = tune.Tuner.restore(path, trainable='train_raytune')

Get the report from Grid Search to CSV file.

In [ ]:
print("🎉[INFO] Training is done!")
print("Best config is:", result.get_best_result().config)
print("Best result is:", result.get_best_result())
df = result.get_dataframe()
df.to_csv('/mnt/d/TA-course-work/Image_processing-2024-solution/Lab6_Hyperparameter-Tuning/csv_results/result.csv', index=False)

# ray.shutdown()

🎉[INFO] Training is done!
Best config is: {'optimizer': 'SGD', 'lr': 0.001, 'batch_size': 16, 'num_epochs': 2}
Best result is: Result(
  metrics={'train_loss': 0.02324574246792646, 'val_loss': 0.020202363561673445, 'val_psnr': 19.170531128023335, 'val_ssim': 0.4798950263451028},
  path='/home/nouzen/ray_results/train_raytune_2024-09-11_13-28-54/train_raytune_1e150_00001_1_batch_size=16,lr=0.0010,num_epochs=2,optimizer=SGD_2024-09-11_13-28-54',
  filesystem='local',
  checkpoint=None
)


---

Train the Autoencoder models using the best hyperparameter set obtained from the grid search.

In [ ]:
torch.manual_seed(4912)
data_dir = '/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba'

files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files]
# Split the dataset into training and testing sets
train_files, test_files = train_test_split(
    files, test_size=0.3, shuffle=True, random_state=2024)
print(files[0])
train_dataset = CustomImageDataset(train_files,
                            resize=132,
                            pad=((132,132),(132,132),(0,0)),
                            padding_mode='reflect',
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
test_dataset = CustomImageDataset(test_files,
                            resize=132,
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
testloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=16)

/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba/000001.jpg


In [ ]:
### START CODE HERE ###
checkpoint_path = 'AE'
# writer = SummaryWriter(f'{checkpoint_path}/logs')
writer = None
model = Autoencoder()
opt = optim.Adam(model.parameters(), lr=0.0008)
loss_fn = nn.MSELoss()
train(model, opt, loss_fn, trainloader, testloader, epochs=10,checkpoint_path='AE',writer=writer, device='cuda')
### END CODE HERE ###

🤖Training on cuda


📄Testing: 100%|██████████| 563/563 [01:09<00:00,  8.09batch/s, loss=0.011, psnr=19, ssim=0.42]     


Summary :
	Train	avg_loss: 0.0179773902666655
	Test	avg_loss: 0.01133054379228218 
		PSNR : 19.155079327363268 
		SSIM : 0.4794327515417344


📄Testing: 100%|██████████| 563/563 [01:10<00:00,  7.94batch/s, loss=0.00833, psnr=19.1, ssim=0.432]


Summary :
	Train	avg_loss: 0.009480006061849004
	Test	avg_loss: 0.008582923816910698 
		PSNR : 19.17027251120185 
		SSIM : 0.4797305389574244


...

Summary :
	Train	avg_loss: 0.00545080413194962
	Test	avg_loss: 0.006241338310157798 
		PSNR : 19.150598401203457 
		SSIM : 0.4791720552550986


📄Testing: 100%|██████████| 563/563 [01:10<00:00,  8.04batch/s, loss=0.0058, psnr=18.8, ssim=0.547] 


Summary :
	Train	avg_loss: 0.005217048478038288
	Test	avg_loss: 0.005196291017867161 
		PSNR : 19.15970750018031 
		SSIM : 0.4795647191415526


📄Testing: 100%|██████████| 563/563 [01:10<00:00,  8.03batch/s, loss=0.00484, psnr=18.8, ssim=0.392]


Summary :
	Train	avg_loss: 0.005036360299234582
	Test	avg_loss: 0.004626257824802483 
		PSNR : 19.167272749150325 
		SSIM : 0.47955706879996857


In [ ]:
# writer = SummaryWriter('runs/customVGG16')
model = Autoencoder()
# train(model.cuda(),opt,loss_fn,train_loader,test_loader,epochs=10,writer=writer,device='cuda')
model.load_state_dict(torch.load('/mnt/d/TA-course-work/Image_processing-2024-solution/Lab6_Hyperparameter-Tuning/AE/model_ep0.pth')['model'])
model.to('cuda')
model.eval()

Autoencoder(
  (downsampling_blocks): ModuleList(
    (0): DownSamplingBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): DownSamplingBlock(
      (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): DownSamplingBlock(
      (conv): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)


Use the `FeatureExtractor()` class and `visualize_feature_map()` function to visualize the feature map of ***ALL*** layers of the Convolution Feature Extractor part. Then, save it as an image.
<details>
<summary>
<font size="3" color="orange">
<b>Expected output</b>
</font>
</summary>

- layer name : vgg16.features.0<br>


- layer name : features.1<br>

- and so on . . . 
</details>

In [ ]:
import math
class FeatureMapVisualizer:
    def __init__(self, model, layers, save_dir):
        """
        Parameters:
        - model: The PyTorch model
        - layers: A string or list of strings specifying the layer names to visualize
        - save_dir: Directory to save the output feature map images
        """
        self.model = model
        self.layers = layers if isinstance(layers, list) else [layers]
        self.activations = {}
        self.save_dir = save_dir

        os.makedirs(self.save_dir, exist_ok=True)

        self._register_hooks()

    def _register_hooks(self):
        for name, layer in self.model.named_modules():
            if name in self.layers:
                layer.register_forward_hook(self._hook_fn(name))

    def _hook_fn(self, layer_name):
        def hook(module, input, output):
            print(f'Hooking layer: {layer_name}')
            self.activations[layer_name] = output.detach()
        return hook

    def visualize(self, input_paths):
        """
        Pass an input tensor through the model and visualize the activations.
        
        Parameters:
        - input_paths: List of image paths to pass through the model
        """
        
        for img_path in input_paths:
            self.model(img_path)

            for layer_name, activation in self.activations.items():
                print(f'Visualizing and saving layer: {layer_name}')
                self._save_feature_maps(activation, layer_name)

    def _save_feature_maps(self, activation, layer_name):
        num_channels = activation.shape[1]
        grid_size = math.ceil(math.sqrt(num_channels))
        num_rows, num_cols = grid_size, grid_size
        
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15),dpi=400)
        fig.suptitle(f'Feature Maps from Layer: {layer_name}', fontsize=10)

        for i in range(num_rows * num_cols):
            row = i // num_cols
            col = i % num_cols
            ax = axes[row, col] if num_rows > 1 else axes[col]
            
            if i < num_channels:
                feature_map = activation[0, i].cpu().numpy()
                ax.imshow(feature_map, cmap='gray')
                
                min_val, max_val = feature_map.min(), feature_map.max()
                ax.set_title(f'ch {i}\n{min_val:.2f} - {max_val:.2f}', fontsize=5)
            else:
                ax.axis('off')
            
            ax.axis('off') 
        plt.tight_layout()
        save_path = os.path.join(self.save_dir, f'{layer_name.replace(".", "_")}.png')
        plt.savefig(save_path)
        plt.close(fig)

In [ ]:
layer_names = []
for layer_name, layer in model.named_modules():
    if layer_name == '':
        continue
    # print(layer_name)
    # print(layer)
    layer_names.append(layer_name)
    # print('-------------------')
print(layer_names)
print(len(layer_names))

['downsampling_blocks', 'downsampling_blocks.0', 'downsampling_blocks.0.conv', 'downsampling_blocks.0.bn', 'downsampling_blocks.0.relu', 'downsampling_blocks.0.pool', 'downsampling_blocks.1', 'downsampling_blocks.1.conv', 'downsampling_blocks.1.bn', 'downsampling_blocks.1.relu', 'downsampling_blocks.1.pool', 'downsampling_blocks.2', 'downsampling_blocks.2.conv', 'downsampling_blocks.2.bn', 'downsampling_blocks.2.relu', 'downsampling_blocks.2.pool', 'downsampling_blocks.3', 'downsampling_blocks.3.conv', 'downsampling_blocks.3.bn', 'downsampling_blocks.3.relu', 'downsampling_blocks.3.pool', 'upsampling_blocks', 'upsampling_blocks.0', 'upsampling_blocks.0.conv', 'upsampling_blocks.0.bn', 'upsampling_blocks.0.relu', 'upsampling_blocks.0.upsample', 'upsampling_blocks.1', 'upsampling_blocks.1.conv', 'upsampling_blocks.1.bn', 'upsampling_blocks.1.relu', 'upsampling_blocks.1.upsample', 'upsampling_blocks.2', 'upsampling_blocks.2.conv', 'upsampling_blocks.2.bn', 'upsampling_blocks.2.relu', 'ups

In [ ]:
batch, labels = next(iter(testloader))
batch[0].shape

input_image = batch[0].unsqueeze(dim=0).to('cuda')
print(input_image.shape)
visualizer = FeatureMapVisualizer(model, layer_names,save_dir='feature_map_result')
visualizer.visualize([input_image])

torch.Size([1, 3, 128, 128])
Hooking layer: conv_in
Hooking layer: downsampling_blocks.0.conv
Hooking layer: downsampling_blocks.0.bn
Hooking layer: downsampling_blocks.0.relu
Hooking layer: downsampling_blocks.0.pool
Hooking layer: downsampling_blocks.0
Hooking layer: downsampling_blocks.1.conv
Hooking layer: downsampling_blocks.1.bn
Hooking layer: downsampling_blocks.1.relu
Hooking layer: downsampling_blocks.1.pool
Hooking layer: downsampling_blocks.1
Hooking layer: downsampling_blocks.2.conv
Hooking layer: downsampling_blocks.2.bn
Hooking layer: downsampling_blocks.2.relu
Hooking layer: downsampling_blocks.2.pool
Hooking layer: downsampling_blocks.2
Hooking layer: downsampling_blocks.3.conv
Hooking layer: downsampling_blocks.3.bn
Hooking layer: downsampling_blocks.3.relu
Hooking layer: downsampling_blocks.3.pool
Hooking layer: downsampling_blocks.3
Hooking layer: upsampling_blocks.0.conv
Hooking layer: upsampling_blocks.0.bn
Hooking layer: upsampling_blocks.0.relu
Hooking layer: ups

---
## **Hyperparameter Random Search with Raytune**

In [ ]:
ray.shutdown()
ray.init(num_gpus=1)

architecture = [[32, 64, 128], [64, 128, 256], [64, 128, 256, 512]]
opts = ["Adam", "SGD"]

config = {
    'architecture': tune.choice(architecture),    
    'optimizer': tune.choice(opts),               
    'lr': tune.uniform(1e-4, 1e-2),               
    'batch_size': tune.randint(16, 33),           
    'num_epochs': tune.randint(10, 101),          
}


tuner = tune.Tuner( 
    tune.with_resources(train_raytune, resources={"gpu": 0.5}),
    tune_config=tune.TuneConfig(
        metric="val_psnr",
        mode="max",
        num_samples=10,
    ),
    param_space=config,
)
result = tuner.fit()

2024-09-11 21:00:29,052	WARNING tune.py:229 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2024-09-11 21:00:29,054	WARNING experiment_state.py:205 -- Experiment state snapshotting has been triggered multiple times in the last 5.0 seconds. A snapshot is forced if `CheckpointConfig(num_to_keep)` is set, and a trial has checkpointed >= `num_to_keep` times since the last snapshot.
You may want to consider increasing the `CheckpointConfig(num_to_keep)` or decreasing the frequency of saving checkpoints.
You can suppress this error by setting the environment variable TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S to a smaller value than the current threshold (5.0).
2024-09-11 21:00:29,056	INFO tune.py:1016 -- Wrote the latest version of all result files and experiment state to '/home/nouzen/ray_results/train_raytune_2024-09-11_21-00-04' in

In [ ]:
print("🎉[INFO] Training is done!")
print("Best config is:", result.get_best_result().config)
print("Best result is:", result.get_best_result())
df = result.get_dataframe()
df.to_csv('/mnt/d/TA-course-work/Image_processing-2024-solution/Lab6_Hyperparameter-Tuning/csv_results/result.csv', index=False)

# ray.shutdown()

---

Train the Autoencoder models using the best hyperparameter set obtained from the random search.

In [ ]:
torch.manual_seed(4912)
data_dir = '/mnt/d/TA-course-work/Image_processing-2024-solution/Lab5_CNN/data/img_align_celeba'

files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files]
# Split the dataset into training and testing sets
train_files, test_files = train_test_split(
    files, test_size=0.3, shuffle=True, random_state=2024)
print(files[0])
train_dataset = CustomImageDataset(train_files,
                            resize=132,
                            pad=((132,132),(132,132),(0,0)),
                            padding_mode='reflect',
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
test_dataset = CustomImageDataset(test_files,
                            resize=132,
                            gauss_blur=True,
                            gauss_noise=True,
                            center_crop=128,
                            p=0.5
                            )
trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
testloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=16)

In [ ]:
### START CODE HERE ###
checkpoint_path = 'AE'
# writer = SummaryWriter(f'{checkpoint_path}/logs')
writer = None
model = Autoencoder()
opt = optim.Adam(model.parameters(), lr=0.0008)
loss_fn = nn.MSELoss()
train(model, opt, loss_fn, trainloader, testloader, epochs=10,checkpoint_path='AE',writer=writer, device='cuda')
### END CODE HERE ###

In [ ]:
# writer = SummaryWriter('runs/customVGG16')
model = Autoencoder()
# train(model.cuda(),opt,loss_fn,train_loader,test_loader,epochs=10,writer=writer,device='cuda')
model.load_state_dict(torch.load('/mnt/d/TA-course-work/Image_processing-2024-solution/Lab6_Hyperparameter-Tuning/AE/model_ep0.pth')['model'])
model.to('cuda')
model.eval()

Use the `FeatureExtractor()` class and `visualize_feature_map()` function to visualize the feature map of ***ALL*** layers of the Convolution Feature Extractor part. Then, save it as an image.
<details>
<summary>
<font size="3" color="orange">
<b>Expected output</b>
</font>
</summary>

- layer name : vgg16.features.0<br>
![vgg16.features.0.png](attachment:vgg16.features.0.png)

- layer name : features.1<br>
![vgg16.features.1.png](attachment:vgg16.features.1.png)
- and so on . . . 
</details>

In [ ]:
layer_names = []
for layer_name, layer in model.named_modules():
    if layer_name == '':
        continue
    # print(layer_name)
    # print(layer)
    layer_names.append(layer_name)
    # print('-------------------')
print(layer_names)
print(len(layer_names))

In [ ]:
batch, labels = next(iter(testloader))
batch[0].shape

input_image = batch[0].unsqueeze(dim=0).to('cuda')
print(input_image.shape)
visualizer = FeatureMapVisualizer(model, layer_names,save_dir='feature_map_result')
visualizer.visualize([input_image])

---

# Questions

1. How many combinations of hyperparameter values (trials) were evaluated during the hyperparameter tuning process?
2. What are the top 3 best parameters and their corresponding tuning results for the model?
3. Analyze and compare the similarities and differences between the top 3 parameters in terms of model architecture, loss, performance, etc.



